 1. Загрузите данные об описаниях вакансий и соответствующих годо
вых зарплатах из файла salary-train.csv.

In [ ]:
import pandas as pd

train = pd.read_csv('salary-train.csv')
test = pd.read_csv('salary-test-mini.csv')


In [ ]:
train

,FullDescription,LocationNormalized,ContractTime,SalaryNormalized
0,international sales manager london k ...,London,permanent,33000
1,an ideal opportunity for an individual that ha...,London,permanent,50000
2,online content and brand manager luxury reta...,South East London,permanent,40000
3,a great local marketleader is seeking a perman...,Dereham,permanent,22500
4,registered nurse rgn nursing home for young...,Sutton Coldfield,nan,20355
...,...,...,...,...
59995,as a result of continued growth first class s...,Whitley Bay,contract,26400
59996,php mvc web developer macclesfieldcirca ...,Macclesfield,permanent,26000
59997,staff nurse nursing home baldock white recru...,Baldock,nan,24500
59998,this is one of the best agency side opportunit...,The City,permanent,65000


In [ ]:
test

,FullDescription,LocationNormalized,ContractTime,SalaryNormalized
0,we currently have a vacancy for an hr project ...,Milton Keynes,contract,NaN
1,a web developer opportunity has arisen with an...,Manchester,permanent,NaN


 2. Проведите предобработку:
 Приведите тексты к нижнему регистру.
 Замените все, кроме букв и цифр, на пробелы это облегчит дальнейшее разделение текста на слова. Для такой замены в строке text подходит следующий вызов:
 re . sub( ’[^a zAZ0 9]’ , ’␣’ , text .lower()) Примените TfidfVectorizer для преобразования текстов в векторы признаков. Оставьте только те слова, которые встречаются хотя бы в 5 объектах (параметр min_df у TfidfVectorizer).
 Замените пропуски в столбцах LocationNormalized и ContractTime на специальную строку ’nan’. Код для этого был приведен выше.
 Примените DictVectorizer для получения one-hot-кодирования признаков LocationNormalized и ContractTime.
 Объедините все полученные признаки воднуматрицу"объекты признаки". Обратите внимание, что матрицы для текстов и категориальных признаков являются разреженными. Для объединения их столбцов нужновоспользоваться функцией scipy.sparse.hstack.

In [ ]:
import re

def clean_text(text):
    return re.sub('[^a-zA-Z0-9]', ' ', text.lower())

train['FullDescription'] = train['FullDescription'].apply(clean_text)
test['FullDescription'] = test['FullDescription'].apply(clean_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(min_df=5)
X_train_text = vectorizer.fit_transform(train['FullDescription'])
X_test_text = vectorizer.transform(test['FullDescription'])


In [ ]:
for col in ['LocationNormalized', 'ContractTime']:
    train[col] = train[col].fillna('nan')
    test[col] = test[col].fillna('nan')


In [ ]:
from sklearn.feature_extraction import DictVectorizer

enc = DictVectorizer()
X_train_categ = enc.fit_transform(train[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_test_categ = enc.transform(test[['LocationNormalized', 'ContractTime']].to_dict('records'))


 3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная записана в столбце SalaryNormalized.

In [ ]:
from scipy.sparse import hstack

X_train = hstack([X_train_text, X_train_categ])
X_test = hstack([X_test_text, X_test_categ])


In [ ]:
from sklearn.linear_model import Ridge

y_train = train['SalaryNormalized']
model = Ridge(alpha=1)
model.fit(X_train, y_train)


Ridge(alpha=1)

 4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv.
 Значения полученных прогнозов являются ответом на задание. Укажите их через пробел

In [ ]:
predictions = model.predict(X_test)
predictions

array([56573.47582704, 37194.9120177 ])